In [ ]:
import ollama
import uuid
from phoenix.otel import register, using_attributes
from openai import OpenAI

in another window run pheonix serve 

In [ ]:
tracer_provider = register(
    project_name="my-llm-app",
    auto_instrument=True,
)

tracer = tracer_provider.get_tracer(__name__)

In [ ]:
span_cm = tracer.start_as_current_span("notebook_chat_session")
parent_span = span_cm.__enter__()

In [ ]:
messages = [{
    'role': 'user',
    'content': 'helloooo r u there?'
}]

in another window start ollama by running: `ollama serve`


Note: this might not be required depending on how you are running ollama, so if you get an error, just try the next line anyways.. Or update if you are using Docker to use the container name and ensure those containers can talk to each other.

In [ ]:
client = OpenAI(
    base_url = 'http://127.0.0.1:11434/v1',
    api_key='ollama', # required, but unused
)

In [ ]:
response = client.chat.completions.create(
    model="llama2",
    messages=messages,
)

In [ ]:
response.choices[0].message.content

In [ ]:
span_cm.__exit__(None, None, None)

### Privacy Evaluation Example

Let's build a small classification example around if private information was revealed in chat or requested, so that you could evaluate or enhance privacy-based guardrails you might be using.

In [ ]:
system_prompt = """
You are a customer service assistant. 
You specifically DO NOT manage person-related data for customers, but instead 
answer frequently asked questions and help with product selection. 

Our list of products is as follows:
- Blue T-shirts
- Red T-shirts
- Purple T-shirts
- I heart Privacy buttons

If a user asks about another product, please politely explain that these are our products now 
and to come back soon if there are any updates.

If a user shares payment, address, or any order details, please do not repeat those details.
Ask them to contact sales at sales@probablyprivate.com. 
"""


In [ ]:
def build_conversation(client, session_id, user_prompt, conversation):
    with using_attributes(session_id=session_id):
        conversation.append(
            {'role': 'user', 
             'content': user_prompt})
        response = client.chat.completions.create(
            model="llama2",
            messages=conversation)
        conversation.append(
            {'role': 'assistant', 
             'content': response.choices[0].message.content})
    print(response.choices[0].message.content)
    return conversation

In [ ]:
def iterate_convo(tracer, client, system_prompt, stop_word="quit"):
    session_id = str(uuid.uuid4())
    span_cm = tracer.start_as_current_span("session_{}".format(session_id))
    parent_span = span_cm.__enter__()
    conversation = [
        {'role': 'system',
         'content': system_prompt},
    ]
    user_input = input(">")
    while user_input != stop_word:
        conversation = build_conversation(client, session_id, user_input, conversation)
        user_input = input(">")
    span_cm.__exit__(None, None, None)

In [ ]:
iterate_convo(tracer, client, system_prompt)